In [1]:
from openai import OpenAI

# 初始化客户端（如果你用 vLLM 本地部署，记得换成对应 base_url）
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")


def multi_query_rewrite(query: str, n: int = 5, model: str = "gpt-4o-mini"):
    """
    输入 query，返回多个改写后的 query
    """
    prompt = f"""
你是一个查询改写助手。请根据用户的问题，生成 {n} 个不同但相关的检索查询，
保证覆盖更多的相关语义和表述方式。

用户问题: "{query}"
请直接输出改写后的查询，每个一行。
"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    rewrites = [
        line.strip("-• ").strip()
        for line in response.choices[0].message.content.split("\n")
        if line.strip()
    ]
    return rewrites[:n]

In [3]:
query = "人工智能对未来就业的影响"
rewrites = multi_query_rewrite(query, n=5, model="./Models/Qwen2.5-14B-Instruct")

print("原始 Query:", query)
print("\n改写后的 Queries:")
for i, q in enumerate(rewrites, 1):
    print(f"{i}. {q}")

原始 Query: 人工智能对未来就业的影响

改写后的 Queries:
1. 人工智能对就业的未来影响
2. 未来就业趋势与人工智能的关系
3. 人工智能如何改变未来的就业市场
4. 探讨人工智能对未来工作岗位的影响
5. 分析人工智能技术对就业市场的长远影响


In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

def research_rewrite(question, n_variants=3):
    """
    研究版的 rewrite: 结合多路改写 + 问题拆解
    """
    prompt = f"""
你是一个信息检索专家。给定一个用户问题：
"{question}"

请你生成 {n_variants} 个不同的查询，每个查询必须满足以下约束：
1. 至少包含 1 个 **语义扩展**的改写（multi-query），即保持问题核心但换不同角度/不同表达。
2. 至少包含 1 个 **子问题拆解**（decomposition），即把复杂问题拆成具体可检索的子问题。
3. 至少包含 1 个 **对立/反向视角**，保证检索覆盖不同立场。

输出时用 JSON 格式返回，字段为：
[
  {{ "type": "multi-query", "query": "..." }},
  {{ "type": "decomposition", "query": "..." }},
  {{ "type": "decomposition", "query": "..." }},
  {{ "type": "multi-query", "query": "..." }},
  {{ "type": "contrastive", "query": "..." }}
]
    """
    response = client.chat.completions.create(
        model="./Models/Qwen2.5-14B-Instruct",  # 你本地跑的模型名，替换成合适的
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


def rewrite_pipeline(questions, model="./Models/Qwen2.5-14B-Instruct"):
    """
    研究版 rewrite pipeline
    输入：问题列表
    输出：每个问题的多路改写（去重后）
    """
    all_results = {}
    for q in questions:
        rewrites = research_rewrite(q)
        all_results[q] = rewrites
    return all_results

In [17]:
questions = [
    "烧水"
]

results = rewrite_pipeline(questions)

import pprint
pprint.pprint(results)


{'烧水': '```json\n'
       '[\n'
       '  {\n'
       '    "type": "multi-query",\n'
       '    "query": "如何快速煮沸一壶水"\n'
       '  },\n'
       '  {\n'
       '    "type": "decomposition",\n'
       '    "query": "使用电热水壶烧水需要多长时间"\n'
       '  },\n'
       '  {\n'
       '    "type": "decomposition",\n'
       '    "query": "在没有电力的情况下如何烧开一壶水"\n'
       '  },\n'
       '  {\n'
       '    "type": "multi-query",\n'
       '    "query": "怎样有效地节约能源来烧水"\n'
       '  },\n'
       '  {\n'
       '    "type": "contrastive",\n'
       '    "query": "为什么有些人认为煮沸水是不必要的"\n'
       '  }\n'
       ']\n'
       '```'}
